# ModelV1 训练流程验证

这个 Notebook 用真实的 ModelV1 数据、离线 DECA 特征和损失函数执行一段端到端训练/验证流程。它还会用 forward hooks 显示 face、eye、crop/camera、scene、fusion 与 UV head 的输入输出维度。

默认只运行 1 个 epoch 且关闭 W&B，用于快速验证线路；将 `USE_CONFIG_EPOCHS` 或 `ENABLE_WANDB` 改为 `True` 可以改用 YAML 的训练配置。

## 1. 导入与路径

请从项目根目录启动 Jupyter，或直接运行此单元。代码会自动把项目根目录加入 Python 导入路径。

In [ ]:
from __future__ import annotations

# Windows 下某些 Conda 环境会同时加载 libomp 和 libiomp5；必须在导入 NumPy/PyTorch 前设置。
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import copy
import random
import sys
import time
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import torch
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'modelv1').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from modelv1.data.dataset import (
    build_modelv1_dataloaders,
    get_uv_target_normalizer,
)
from modelv1.losses import UVLossConfig, UVRegressionLoss
from modelv1.model import ModelV1, ModelV1Config

print(f'项目根目录: {PROJECT_ROOT}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA 可用: {torch.cuda.is_available()}')

## 2. 读取训练配置

`NOTEBOOK_EPOCHS=1` 只用于检查流程是否可运行，不会修改 YAML 文件。当前 YAML 中的 epoch 数仅在 `USE_CONFIG_EPOCHS=True` 时才会生效。

In [ ]:
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'modelv1' / 'train_random_80_20_100.yaml'
NOTEBOOK_EPOCHS = 1
USE_CONFIG_EPOCHS = False
ENABLE_WANDB = False
SAVE_BEST_CHECKPOINT = True

with CONFIG_PATH.open('r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

config = copy.deepcopy(config)
if not USE_CONFIG_EPOCHS:
    config['training']['epochs'] = NOTEBOOK_EPOCHS
config['logging']['wandb']['enabled'] = ENABLE_WANDB

def resolve_project_path(value: str | Path) -> Path:
    path = Path(value)
    return path if path.is_absolute() else PROJECT_ROOT / path

print(f'配置文件: {CONFIG_PATH}')
print(f"本次 epoch: {config['training']['epochs']}")
print(f"数据划分: {config['data']['split_mode']}")
print(f"W&B: {config['logging']['wandb']['enabled']}")

## 3. 固定随机性、选择设备并构建 DataLoader

随机 4:1 划分会从数据集 3、4、5 的样本中确定性地划出训练集和验证集。UV 归一化统计量只在训练子集上拟合。

In [ ]:
def seed_everything(seed: int, deterministic: bool) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = not deterministic

seed_everything(config['experiment']['seed'], config['experiment']['deterministic'])
requested_device = config['training']['device']
device = torch.device('cuda' if requested_device == 'auto' and torch.cuda.is_available() else 'cpu' if requested_device == 'auto' else requested_device)
if device.type == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('配置要求 CUDA，但当前 PyTorch 未检测到可用 CUDA。')

data_config = config['data']
train_loader, val_loader = build_modelv1_dataloaders(
    csv_path=resolve_project_path(data_config['csv_path']),
    split_mode=data_config['split_mode'],
    all_datasets=data_config['all_datasets'],
    val_ratio=data_config['val_ratio'],
    split_seed=data_config['split_seed'],
    batch_size=data_config['batch_size'],
    num_workers=data_config['num_workers'],
    pin_memory=data_config['pin_memory'],
    normalize_images=data_config['normalize_images'],
    load_face_image=data_config['load_face_image'],
    deca_cache_path=resolve_project_path(data_config['deca_cache_path']),
    require_deca_features=True,
    normalize_uv_targets=True,
)
normalizer = get_uv_target_normalizer(train_loader.dataset)
if normalizer is None:
    raise RuntimeError('未获得 UV 目标归一化器。')

print(f'设备: {device}')
print(f'训练样本: {len(train_loader.dataset)}, 验证样本: {len(val_loader.dataset)}')
print(f'训练 batch 数: {len(train_loader)}, 验证 batch 数: {len(val_loader)}')
print(f'UV 均值 [mm]: {normalizer.mean_mm.tolist()}')
print(f'UV 标准差 [mm]: {normalizer.std_mm.tolist()}')

## 4. 检查 DataLoader 的 batch 维度

其中 `B` 是当前 batch 的样本数。当前模型使用离线缓存的 `deca_feat [B, 236]`，所以当 `load_face_image=false` 时不会读取原始 face 图像。

In [ ]:
batch = next(iter(train_loader))
for name, value in batch.items():
    if torch.is_tensor(value):
        print(f'{name:16s} shape={tuple(value.shape)!s:18s} dtype={str(value.dtype)}')
    elif name != 'paths':
        print(f'{name:16s} type={type(value).__name__}')

assert batch['deca_feat'].shape[-1] == config['model']['deca_feature_dim']
assert batch['uv_target'].shape[-1] == 2
assert torch.allclose(normalizer.denormalize(batch['uv_target']), batch['uv_gt'], atol=1e-4)
print('UV 归一化与反归一化检查通过。')

## 5. 建模并显示各模块输入/输出维度

`fusion_mlp` 的输入是四个分支特征拼接得到的 `[B, 384]`：`128 + 128 + 64 + 64`。UV head 的输出 `[B, 2]` 是归一化坐标，训练后通过 normalizer 转回毫米坐标。

In [ ]:
model_kwargs = dict(config['model'])
for key in ('face_hidden_dims', 'crop_cam_hidden_dims', 'scene_hidden_dims', 'fusion_hidden_dims'):
    model_kwargs[key] = tuple(model_kwargs[key])
model = ModelV1(ModelV1Config(**model_kwargs)).to(device)

def move_batch_to_device(batch_dict: dict[str, object]) -> dict[str, object]:
    return {
        key: value.to(device, non_blocking=True) if torch.is_tensor(value) else value
        for key, value in batch_dict.items()
    }

def shape_of(value: object) -> object:
    if torch.is_tensor(value):
        return tuple(value.shape)
    if isinstance(value, (tuple, list)):
        return [shape_of(item) for item in value]
    if isinstance(value, dict):
        return {key: shape_of(item) for key, item in value.items()}
    return type(value).__name__

module_shapes: dict[str, tuple[object, object]] = {}
handles = []
for module_name in ('face_branch', 'eye_branch', 'crop_cam_branch', 'scene_branch', 'fusion_mlp', 'uv_head'):
    module = getattr(model, module_name)
    def hook(_module, inputs, output, name=module_name):
        module_shapes[name] = (shape_of(inputs), shape_of(output))
    handles.append(module.register_forward_hook(hook))

shape_batch = move_batch_to_device(batch)
model.eval()
with torch.no_grad():
    feature_output = model(shape_batch, return_features=True)
for handle in handles:
    handle.remove()

for module_name, (input_shape, output_shape) in module_shapes.items():
    print(f'{module_name:16s} input={input_shape} -> output={output_shape}')
print('--- returned features ---')
for name, value in feature_output.items():
    print(f'{name:16s} {tuple(value.shape)}')
print('--- fusion contract ---')
print(f"face {tuple(feature_output['face_features'].shape)} + eye {tuple(feature_output['eye_features'].shape)} + "
      f"crop/cam {tuple(feature_output['crop_cam_features'].shape)} + scene {tuple(feature_output['scene_features'].shape)}")
model.train()

## 6. 损失、优化器、学习率调度器

训练损失是目标归一化空间中的逐轴 Smooth L1。`beta_mm=30` 会按训练集的 u/v 标准差换算为对应的归一化阈值；验证指标则始终在毫米空间计算。

In [ ]:
loss_fn = UVRegressionLoss(normalizer, UVLossConfig(**config['loss']))
optimizer_config = config['training']['optimizer']
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=optimizer_config['lr'],
    weight_decay=optimizer_config['weight_decay'],
)
scheduler_config = config['training']['scheduler']
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config['training']['epochs'],
    eta_min=scheduler_config['eta_min'],
)
amp_enabled = bool(config['training']['amp']) and device.type == 'cuda'
scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

def autocast_context():
    return torch.autocast(device_type='cuda', dtype=torch.float16) if amp_enabled else nullcontext()

print(f'优化器: AdamW, lr={optimizer.param_groups[0]["lr"]}')
print(f'调度器: CosineAnnealingLR, T_max={config["training"]["epochs"]}')
print(f'AMP: {amp_enabled}')

## 7. 单 epoch 训练与验证函数

训练阶段包含前向传播、反向传播、AMP 梯度缩放、梯度裁剪与参数更新。验证阶段关闭梯度，仅汇总 loss 和毫米空间误差。

In [ ]:
def run_epoch(loader, training: bool) -> dict[str, float]:
    model.train(training)
    started_at = time.perf_counter()
    loss_sum = 0.0
    sample_count = 0
    predictions = []
    targets_mm = []

    grad_context = torch.enable_grad() if training else torch.no_grad()
    with grad_context:
        for cpu_batch in loader:
            model_batch = move_batch_to_device(cpu_batch)
            uv_target = model_batch['uv_target']
            uv_gt_mm = model_batch['uv_gt']
            batch_size = uv_target.shape[0]

            if training:
                optimizer.zero_grad(set_to_none=True)

            with autocast_context():
                uv_pred = model(model_batch)
                loss = loss_fn(uv_pred, uv_target)

            if training:
                scaler.scale(loss).backward()
                grad_clip_norm = config['training']['grad_clip_norm']
                if grad_clip_norm is not None:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
                scaler.step(optimizer)
                scaler.update()

            loss_sum += loss.detach().float().item() * batch_size
            sample_count += batch_size
            predictions.append(uv_pred.detach().float().cpu())
            targets_mm.append(uv_gt_mm.detach().float().cpu())

    metrics = loss_fn.metrics(torch.cat(predictions), torch.cat(targets_mm))
    elapsed = time.perf_counter() - started_at
    return {
        'loss': loss_sum / sample_count,
        **{name: value.item() for name, value in metrics.items()},
        'seconds': elapsed,
        'samples_per_second': sample_count / elapsed,
    }

## 8. 执行训练、验证、学习率更新与最优 checkpoint 保存

最优模型按 `val/epe_mm`（越小越好）选择。启用 W&B 时，每个 epoch 的 loss、EPE、median EPE、u/v MAE、吞吐和学习率都会被写入当前 W&B run。

In [ ]:
wandb_run = None
if config['logging']['wandb']['enabled']:
    import wandb
    wandb_config = config['logging']['wandb']
    wandb_run = wandb.init(
        project=wandb_config['project'],
        entity=wandb_config['entity'],
        mode=wandb_config['mode'],
        tags=wandb_config['tags'],
        config=config,
    )

checkpoint_dir = PROJECT_ROOT / 'outputs' / 'notebook_training_validation'
checkpoint_dir.mkdir(parents=True, exist_ok=True)
best_checkpoint_path = checkpoint_dir / 'best.pt'
best_val_epe_mm = float('inf')
history = {key: [] for key in ('epoch', 'train_loss', 'val_loss', 'train_epe_mm', 'val_epe_mm')}

for epoch in range(1, config['training']['epochs'] + 1):
    train_metrics = run_epoch(train_loader, training=True)
    val_metrics = run_epoch(val_loader, training=False)
    scheduler.step()

    history['epoch'].append(epoch)
    history['train_loss'].append(train_metrics['loss'])
    history['val_loss'].append(val_metrics['loss'])
    history['train_epe_mm'].append(train_metrics['epe_mm'])
    history['val_epe_mm'].append(val_metrics['epe_mm'])

    if val_metrics['epe_mm'] < best_val_epe_mm:
        best_val_epe_mm = val_metrics['epe_mm']
        if SAVE_BEST_CHECKPOINT:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'normalizer': normalizer.state_dict(),
                'model_config': model_kwargs,
                'best_val_epe_mm': best_val_epe_mm,
            }, best_checkpoint_path)

    print(
        f"epoch {epoch:03d}/{config['training']['epochs']} | "
        f"train loss {train_metrics['loss']:.4f}, EPE {train_metrics['epe_mm']:.2f} mm | "
        f"val loss {val_metrics['loss']:.4f}, EPE {val_metrics['epe_mm']:.2f} mm | "
        f"val median {val_metrics['median_epe_mm']:.2f} mm | "
        f"lr {optimizer.param_groups[0]['lr']:.2e}"
    )

    if wandb_run is not None:
        wandb_run.log({
            'epoch': epoch,
            **{f'train/{key}': value for key, value in train_metrics.items()},
            **{f'val/{key}': value for key, value in val_metrics.items()},
            'learning_rate': optimizer.param_groups[0]['lr'],
            'checkpoint/best_val_epe_mm': best_val_epe_mm,
        })

if wandb_run is not None:
    wandb_run.finish()

if SAVE_BEST_CHECKPOINT:
    print(f'最优 checkpoint: {best_checkpoint_path}')
print(f'最优验证 EPE: {best_val_epe_mm:.2f} mm')

## 9. 绘制训练曲线并查看验证集预测样例

EPE 是二维预测坐标与真实坐标的欧氏距离，单位为毫米。

In [ ]:
import matplotlib.pyplot as plt

if 'history' not in globals():
    raise RuntimeError('请先按顺序运行第 1-8 节；训练单元会创建 history、model 和验证集预测所需变量。')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['epoch'], history['train_loss'], marker='o', label='train')
axes[0].plot(history['epoch'], history['val_loss'], marker='o', label='val')
axes[0].set(title='Smooth L1 loss', xlabel='epoch', ylabel='loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history['epoch'], history['train_epe_mm'], marker='o', label='train')
axes[1].plot(history['epoch'], history['val_epe_mm'], marker='o', label='val')
axes[1].set(title='2D endpoint error', xlabel='epoch', ylabel='EPE (mm)')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

model.eval()
validation_batch = move_batch_to_device(next(iter(val_loader)))
with torch.no_grad():
    predicted_uv_mm = normalizer.denormalize(model(validation_batch).float()).cpu()
ground_truth_uv_mm = validation_batch['uv_gt'].float().cpu()
sample_epe_mm = torch.linalg.vector_norm(predicted_uv_mm - ground_truth_uv_mm, dim=-1)

print(' index | pred_u  pred_v | gt_u    gt_v  | EPE (mm)')
for index in range(min(8, len(predicted_uv_mm))):
    pred = predicted_uv_mm[index]
    target = ground_truth_uv_mm[index]
    print(f'{index:6d} | {pred[0]:7.2f} {pred[1]:7.2f} | {target[0]:7.2f} {target[1]:7.2f} | {sample_epe_mm[index]:8.2f}')

## 下一步

这个 Notebook 用于验证训练线路和张量维度。正式的长时间训练仍建议使用 `scripts/train_modelv1.py`，它会额外维护本地 `train.log`、`metrics.csv`、可恢复 checkpoint 与 W&B 运行记录。